# GoEmotions RoBERTa-large Focal Loss Inference

This notebook loads the public Hugging Face checkpoint and applies the validation-selected thresholds from the released `thresholds.json` file.

- Hugging Face model: https://huggingface.co/AliceYin/goemotions-roberta-large-focal-sota
- Kaggle model artifact: https://www.kaggle.com/models/kevin250304/goemotions-roberta-large-focal-sota/Transformers/roberta-large-focal-seed42
- Training source: https://github.com/Kevin-Li-2025/goemotions-roberta-large-focal


In [ ]:
import json

import torch
from huggingface_hub import hf_hub_download
from transformers import AutoModelForSequenceClassification, AutoTokenizer

HF_MODEL_ID = "AliceYin/goemotions-roberta-large-focal-sota"
KAGGLE_MODEL_URL = (
    "https://www.kaggle.com/models/kevin250304/"
    "goemotions-roberta-large-focal-sota/Transformers/roberta-large-focal-seed42"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(HF_MODEL_ID).to(device)
model.eval()

with open(hf_hub_download(HF_MODEL_ID, "labels.json"), encoding="utf-8") as f:
    labels = json.load(f)["label_names"]
with open(hf_hub_download(HF_MODEL_ID, "thresholds.json"), encoding="utf-8") as f:
    threshold_data = json.load(f)

selected_policy = threshold_data["selected"]
selected_thresholds = threshold_data[selected_policy]
threshold_map = (
    selected_thresholds["per_label"]
    if selected_policy == "global"
    else selected_thresholds
)
thresholds = torch.tensor([threshold_map[label] for label in labels], device=device)

selected_policy, KAGGLE_MODEL_URL


In [ ]:
def predict_emotions(texts, top_k=5):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=160,
    ).to(device)
    with torch.no_grad():
        probs = torch.sigmoid(model(**inputs).logits)

    rows = []
    for text, row_probs in zip(texts, probs):
        thresholded = [
            {"label": label, "score": round(float(prob), 4)}
            for label, prob, threshold in zip(labels, row_probs, thresholds)
            if prob >= threshold
        ]
        top_indices = torch.topk(row_probs, k=min(top_k, len(labels))).indices.tolist()
        top_scores = [
            {"label": labels[i], "score": round(float(row_probs[i]), 4)}
            for i in top_indices
        ]
        rows.append({"text": text, "labels": thresholded, "top_scores": top_scores})
    return rows


In [ ]:
sample_texts = [
    "I finally got this working and I am so relieved.",
    "That was rude and completely unnecessary.",
    "I am not sure how I feel about this yet.",
]

predict_emotions(sample_texts)
